# Model Training and Evaluation
## Objective
This notebook trains multiple regression models using the engineered Uber dataset and compares their performance using MAE, RMSE, and R² Score.
The best-performing model will be selected for deployment.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import plotly.express as px

In [2]:
df = pd.read_csv("../data/processed/uber_feature_engineered.csv")
df.head()

,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_hour,pickup_day,day_of_week,pickup_month,pickup_year,is_weekend,is_peak_hour,trip_distance_km
0,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06+00:00,-73.999817,40.738354,-73.999512,40.723217,1,19,Thursday,3,5,2015,0,1,1.683323
1,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56+00:00,-73.994355,40.728225,-73.994710,40.750325,1,20,Friday,4,7,2009,0,0,2.457590
2,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00+00:00,-74.005043,40.740770,-73.962565,40.772647,1,21,Monday,0,8,2009,0,0,5.036377
3,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21+00:00,-73.976124,40.790844,-73.965316,40.803349,3,8,Friday,4,6,2009,0,1,1.661683
4,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00+00:00,-73.925023,40.744085,-73.973082,40.761247,5,17,Thursday,3,8,2014,0,1,4.475450


In [3]:
df.shape

(195333, 16)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 195333 entries, 0 to 195332
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   key                195333 non-null  str    
 1   fare_amount        195333 non-null  float64
 2   pickup_datetime    195333 non-null  str    
 3   pickup_longitude   195333 non-null  float64
 4   pickup_latitude    195333 non-null  float64
 5   dropoff_longitude  195333 non-null  float64
 6   dropoff_latitude   195333 non-null  float64
 7   passenger_count    195333 non-null  int64  
 8   pickup_hour        195333 non-null  int64  
 9   pickup_day         195333 non-null  str    
 10  day_of_week        195333 non-null  int64  
 11  pickup_month       195333 non-null  int64  
 12  pickup_year        195333 non-null  int64  
 13  is_weekend         195333 non-null  int64  
 14  is_peak_hour       195333 non-null  int64  
 15  trip_distance_km   195333 non-null  float64
dtypes: float64(6)

In [5]:
df.describe()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_hour,day_of_week,pickup_month,pickup_year,is_weekend,is_peak_hour,trip_distance_km
count,195333.000000,195333.000000,195333.000000,195333.000000,195333.000000,195333.000000,195333.000000,195333.000000,195333.000000,195333.000000,195333.000000,195333.000000,195333.000000
mean,11.352350,-73.900111,40.686478,-73.899195,40.685881,1.690093,13.491924,3.049454,6.283849,2011.740873,0.283618,0.341847,5.153867
std,9.804447,2.825267,2.656316,2.820956,2.661708,1.306093,6.514963,1.946840,3.439399,1.861479,0.450755,0.474330,104.352154
min,0.010000,-93.824668,-74.015515,-75.458979,-74.015750,1.000000,0.000000,0.000000,1.000000,2009.000000,0.000000,0.000000,0.000000
25%,6.000000,-73.992267,40.736390,-73.991592,40.735265,1.000000,9.000000,1.000000,3.000000,2010.000000,0.000000,0.000000,1.255851
50%,8.500000,-73.982101,40.753275,-73.980521,40.753723,1.000000,14.000000,3.000000,6.000000,2012.000000,0.000000,0.000000,2.157924
75%,12.500000,-73.968313,40.767537,-73.965314,40.768320,2.000000,19.000000,5.000000,9.000000,2013.000000,1.000000,1.000000,3.911449
max,499.000000,40.808425,48.018760,40.831932,45.031598,6.000000,23.000000,6.000000,12.000000,2015.000000,1.000000,1.000000,8708.233063


In [6]:
drop_columns = [
    "key",
    "pickup_datetime",
    "pickup_day"
]

X = df.drop(
    columns=drop_columns + ["fare_amount"]
)

y = df["fare_amount"]

In [7]:
X.head()

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,pickup_hour,day_of_week,pickup_month,pickup_year,is_weekend,is_peak_hour,trip_distance_km
0,-73.999817,40.738354,-73.999512,40.723217,1,19,3,5,2015,0,1,1.683323
1,-73.994355,40.728225,-73.994710,40.750325,1,20,4,7,2009,0,0,2.457590
2,-74.005043,40.740770,-73.962565,40.772647,1,21,0,8,2009,0,0,5.036377
3,-73.976124,40.790844,-73.965316,40.803349,3,8,4,6,2009,0,1,1.661683
4,-73.925023,40.744085,-73.973082,40.761247,5,17,3,8,2014,0,1,4.475450


In [8]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 195333 entries, 0 to 195332
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   pickup_longitude   195333 non-null  float64
 1   pickup_latitude    195333 non-null  float64
 2   dropoff_longitude  195333 non-null  float64
 3   dropoff_latitude   195333 non-null  float64
 4   passenger_count    195333 non-null  int64  
 5   pickup_hour        195333 non-null  int64  
 6   day_of_week        195333 non-null  int64  
 7   pickup_month       195333 non-null  int64  
 8   pickup_year        195333 non-null  int64  
 9   is_weekend         195333 non-null  int64  
 10  is_peak_hour       195333 non-null  int64  
 11  trip_distance_km   195333 non-null  float64
dtypes: float64(5), int64(7)
memory usage: 17.9 MB


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [10]:
print("Training Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

Training Shape : (156266, 12)
Testing Shape : (39067, 12)


## Baseline Model
Linear Regression is used as the baseline model.
Although simple, it provides an important benchmark against which more complex models can be compared.

In [11]:
lr = LinearRegression()

lr.fit(
    X_train,
    y_train
)

lr_pred = lr.predict(X_test)

In [12]:
def evaluate_model(model_name, y_true, y_pred):
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return {
        "Model": model_name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R² Score": round(r2, 4)
    }

In [13]:
results = []

results.append(
    evaluate_model(
        "Linear Regression",
        y_test,
        lr_pred
    )
)

In [14]:
dt = DecisionTreeRegressor(
    random_state=42
)

dt.fit(
    X_train,
    y_train
)

dt_pred = dt.predict(X_test)

results.append(
    evaluate_model(
        "Decision Tree",
        y_test,
        dt_pred
    )
)

In [15]:
rf = RandomForestRegressor(
    random_state=42,
    n_estimators=100,
    n_jobs=-1
)

rf.fit(
    X_train,
    y_train
)

rf_pred = rf.predict(X_test)

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_pred
    )
)

In [16]:
gb = GradientBoostingRegressor(
    random_state=42
)

gb.fit(
    X_train,
    y_train
)

gb_pred = gb.predict(X_test)

results.append(
    evaluate_model(
        "Gradient Boosting",
        y_test,
        gb_pred
    )
)

In [17]:
results_df = pd.DataFrame(results)
results_df

,Model,MAE,RMSE,R² Score
0,Linear Regression,5.92,9.57,0.0175
1,Decision Tree,2.57,5.29,0.6994
2,Random Forest,1.81,3.68,0.8542
3,Gradient Boosting,1.98,3.85,0.8405


In [18]:
results_df.sort_values(
    by="R² Score",
    ascending=False
)

,Model,MAE,RMSE,R² Score
2,Random Forest,1.81,3.68,0.8542
3,Gradient Boosting,1.98,3.85,0.8405
1,Decision Tree,2.57,5.29,0.6994
0,Linear Regression,5.92,9.57,0.0175


In [19]:
fig = px.bar(
    results_df,
    x="Model",
    y="R² Score",
    color="R² Score",
    text="R² Score",
    title="Model Performance Comparison",
    template="plotly_white"
)

fig.update_layout(
    title_x=0.5
)

fig.show()

# Best Model Selection

Random Forest achieved the highest R² Score while maintaining the lowest MAE and RMSE among all evaluated models.

This model will therefore be used for deployment.

In [20]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

In [21]:
fig = px.bar(
    importance,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Random Forest Feature Importance",
    template="plotly_white"
)

fig.update_layout(
    title_x=0.5
)

fig.show()

In [22]:
comparison = pd.DataFrame({
    "Actual Fare": y_test,
    "Predicted Fare": rf_pred
})

comparison["Error"] = (
    comparison["Actual Fare"] -
    comparison["Predicted Fare"]
)

comparison.head()

,Actual Fare,Predicted Fare,Error
17496,7.0,6.265,0.735
101766,12.0,13.165,-1.165
45624,7.3,9.716,-2.416
120859,14.9,18.261,-3.361
82411,10.1,12.588,-2.488


In [23]:
comparison.describe()

,Actual Fare,Predicted Fare,Error
count,39067.000000,39067.000000,39067.000000
mean,11.326803,11.460839,-0.134036
std,9.652180,8.940698,3.682569
min,2.500000,3.481000,-75.694700
25%,6.000000,6.502000,-1.296000
50%,8.500000,8.695000,-0.386000
75%,12.500000,12.570000,0.690000
max,191.800000,119.084600,100.120600


In [24]:
fig = px.scatter(
    comparison,
    x="Predicted Fare",
    y="Error",
    opacity=0.5,
    title="Residual Plot",
    template="plotly_white"
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="red"
)

fig.show()

## Observation
Residuals are centered around zero
Error variance increases for larger predicted fares.
No strong systematic bias is visible.

## Business Insight
The Random Forest model captures the overall fare pattern well, though prediction uncertainty grows for high-value trips.

In [25]:
fig = px.scatter(
    comparison,
    x="Actual Fare",
    y="Predicted Fare",
    opacity=0.5,
    title="Actual vs Predicted Fare",
    template="plotly_white"
)

fig.add_shape(
    type="line",
    x0=comparison["Actual Fare"].min(),
    y0=comparison["Actual Fare"].min(),
    x1=comparison["Actual Fare"].max(),
    y1=comparison["Actual Fare"].max(),
    line=dict(color="red")
)

fig.show()

## Observation

Most predictions lie close to the ideal diagonal line.

Prediction errors increase for higher fares.

## Business Insight

The model performs well for typical rides but is less accurate for expensive trips, likely because these rides are less frequent in the training data.

In [ ]:
results_df.to_csv(
    "../reports/model_comparison.csv",
    index=False
)

In [ ]:
importance.to_csv(
    "../reports/feature_importance.csv",
    index=False
)